In [1]:
import numpy as np 
from mini_nn.losses import BCELoss, BCEWithLogitLoss, CrossEnthropyLoss

# Binary Cross Enthropy Check


In [ ]:
# --- Simple known value check ---
y_true = np.array([1.0])
y_pred = np.array([0.5])

bce_loss = BCELoss()
loss = bce_loss(y_true, y_pred)

print("Expected loss:", -np.log(0.5))
print("Actual loss:  ", loss)

Expected loss: 0.6931471805599453
Actual loss:   0.6931471805599453


In [3]:
# --- Backward check with a value that is easy to calculate by hand ---
y_true = np.array([1.0])
y_pred = np.array([0.8])

bce_loss = BCELoss()
bce_loss(y_true, y_pred)

grad = bce_loss.backward()

expected_grad = (0.8 - 1) / (0.8 * (1 - 0.8))

print("Expected grad:", expected_grad)
print("Actual grad:  ", grad)

Expected grad: -1.25
Actual grad:   [-1.25]


In [4]:
# --- Good predictions should have lower loss than bad predictions ---
y_true = np.array([1.0, 1.0, 0.0, 0.0])

good_pred = np.array([0.9, 0.8, 0.2, 0.1])
bad_pred  = np.array([0.1, 0.2, 0.8, 0.9])

bce_loss = BCELoss()

good_loss = bce_loss(y_true, good_pred)
bad_loss = bce_loss(y_true, bad_pred)

print("Good predictions loss:", good_loss)
print("Bad predictions loss: ", bad_loss)
print("Good < Bad:", good_loss < bad_loss)

Good predictions loss: 0.16425203348601802
Bad predictions loss:  1.956011502714073
Good < Bad: True


In [5]:
# --- Shape mismatch check ---
y_true = np.array([1.0, 0.0, 1.0])
y_pred = np.array([0.8, 0.2])

bce_loss = BCELoss()

try:
    bce_loss(y_true, y_pred)
except ValueError as e:
    print(e)

y_true and y_pred must be the same shape


# BCEWithLogitLoss Check

In [6]:
# --- BCE(sigmoid(logits)) should match BCEWithLogits(logits) ---

y_true = np.array([1.0, 0.0, 1.0, 0.0])
logits = np.array([2.0, -1.0, 0.5, -2.0])

probabilities = 1 / (1 + np.exp(-logits))

bce = BCELoss()
bce_logits = BCEWithLogitLoss()

loss_bce = bce(y_true, probabilities)
loss_logits = bce_logits(y_true, logits)

print("BCE loss:           ", loss_bce)
print("BCEWithLogits loss: ", loss_logits)
print("Close:", np.allclose(loss_bce, loss_logits))

BCE loss:            0.26029867344606866
BCEWithLogits loss:  0.2602986734460687
Close: True


In [7]:
# --- Extreme logits stability check ---
y_true = np.array([1.0, 0.0])
logits = np.array([100.0, -100.0])

bce_logits = BCEWithLogitLoss()
loss = bce_logits(y_true, logits)

print("Loss:", loss)
print("Finite:", np.isfinite(loss))

Loss: 0.0
Finite: True


# Multi-Class Cross Enthropy

In [ ]:
ce_loss = CrossEnthropyLoss()

# --- simple known value check ---
y_true = np.array([
    [0,0,0,1],
    [0,1,0,0],
    [0,0,0,1],
    [1,0,0,0],
    [0,0,1,0]
])
print(y_true.shape)

y_pred_raw = np.random.uniform(size=y_true.shape)
y_pred = y_pred_raw / np.sum(y_pred_raw, axis=1, keepdims=True)

(5, 4)


In [16]:
loss = ce_loss(y_true, y_pred)
print(loss)

1.8220918824588803


In [ ]:
print(np.sum(y_pred, axis=1))

expected_loss = -np.mean(
    np.sum(y_true * np.log(y_pred), axis=1)
)

print(f'Expected: {expected_loss}')
print(f'Actual:   {loss}')
print(f'Close:    {np.allclose(loss, expected_loss)}')

[1. 1. 1. 1. 1.]
Expected: 1.8220918824588803
Actual:   1.8220918824588803
Close:    True


In [36]:
# --- back propagation check ---
grad = ce_loss.backward()
expected_grad = -(y_true / y_pred) / y_true.shape[0]

print("Gradient shape:", grad.shape)
print("Expected shape:", expected_grad.shape)

print("Close:", np.allclose(grad, expected_grad))

print("Actual gradient:")
print(grad)

print("Expected gradient:")
print(expected_grad)

Gradient shape: (5, 4)
Expected shape: (5, 4)
Close: True
Actual gradient:
[[-0.         -0.         -0.         -1.07822752]
 [-0.         -3.10865652 -0.         -0.        ]
 [-0.         -0.         -0.         -0.52977115]
 [-2.88472803 -0.         -0.         -0.        ]
 [-0.         -0.         -0.56532202 -0.        ]]
Expected gradient:
[[-0.         -0.         -0.         -1.07822752]
 [-0.         -3.10865652 -0.         -0.        ]
 [-0.         -0.         -0.         -0.52977115]
 [-2.88472803 -0.         -0.         -0.        ]
 [-0.         -0.         -0.56532202 -0.        ]]
